In [2]:
print(df.columns.tolist())

['Duration_in_month', 'Credit_amount', 'Installment_rate_in_percentage_of_disposable_income', 'Present_residence_since', 'Age_in_years', 'Number_of_existing_credits_at_this_bank', 'Number_of_people_being_liable_to_provide_maintenance_for', 'Status_of_existing_checking_account_< 0 DM', 'Status_of_existing_checking_account_>= 200 DM / salary assignments for at least 1 year', 'Status_of_existing_checking_account_no checking account', 'Credit_history_critical account / other credits existing (not at this bank)', 'Credit_history_delay in paying off in the past', 'Credit_history_existing credits paid back duly till now', 'Credit_history_no credits taken / all credits paid back duly', 'Purpose_car (new)', 'Purpose_car (used)', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_others', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_retraining', 'Savings_account_bonds_500 <= ... < 1000 DM', 'Savings_account_bonds_< 100 DM', 'Savings_account_bonds_>= 10

In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load the processed data
data_path = '../data/processed/german_credit_ml_ready.csv'
df = pd.read_csv(data_path)

# 2. Separate Features (X) from the Target (y)
# 'Risk' is what we are trying to predict
X = df.drop('Risk_1', axis=1)
y = df['Risk_1']

# 3. Train-Test Split (80% for training, 20% for testing)
# random_state ensures we get the exact same split every time we run this
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (800, 48)
Testing data shape: (200, 48)


In [5]:
# 1. Initialize the Random Forest Model
# class_weight='balanced' forces the model to pay more attention to the minority class (Bad loans)
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    class_weight='balanced'
)

# 2. Train the model (This is where the actual "learning" happens)
print("Training the Random Forest model...")
rf_model.fit(X_train, y_train)
print("Training complete!")

# 3. Make predictions on our hidden 20% test set
y_pred = rf_model.predict(X_test)

# 4. Evaluate how well it did
print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred))

Training the Random Forest model...
Training complete!

--- CONFUSION MATRIX ---
[[128  12]
 [ 36  24]]

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

       False       0.78      0.91      0.84       140
        True       0.67      0.40      0.50        60

    accuracy                           0.76       200
   macro avg       0.72      0.66      0.67       200
weighted avg       0.75      0.76      0.74       200



In [6]:
# Save the trained model to our artifacts folder
model_path = '../model_artifacts/random_forest_model.pkl'
joblib.dump(rf_model, model_path)

print(f"Success! Model saved to: {model_path}")

# (Optional) Save the feature names so the API knows exactly what order the columns should be in
features_path = '../model_artifacts/model_features.pkl'
joblib.dump(list(X.columns), features_path)
print(f"Success! Feature list saved to: {features_path}")

Success! Model saved to: ../model_artifacts/random_forest_model.pkl
Success! Feature list saved to: ../model_artifacts/model_features.pkl
